In [42]:
!pip install -q openpyxl joblib


In [43]:
import pandas as pd

csv_path = '/kaggle/input/datasets/odusamioluwadamilare/wideformat-dataset/formatted_wide.csv'
df = pd.read_csv(csv_path)

print(df.shape)
df.head()

(1017, 14)


,record_id,behavioral_profile.avg_star_rating,behavioral_profile.rating_consistency,behavioral_profile.sentiment_bias,behavioral_profile.verbal_style,history[0],history[1],history[2],history[3],history[4],nigerian_adaptation.persona_type,nigerian_adaptation.suggested_markers[0],nigerian_adaptation.suggested_markers[1],nigerian_adaptation.suggested_markers[2]
0,-2nNiAnDaSbQayynsqgL6Q,3.00,Stable,Critical,detailed,"Jugs catered our company picnic (at work), and...","I paid cash for a used car in March. In April,...",Met some friends here after work. It was a goo...,"YUM. I am stuffed as I write this, and I'll pr...",I know a lot of people compare WG Grinders to ...,Lagos Consumer Proxy,sha,abeg,standard
1,-B-QEUESGWHPE_889WJaeg,3.40,Stable,Critical,detailed,"Food is good, manager is great, kitchen timing...",The Granada is a beautifully remodeled theater...,"Epiphany reopened as Hob Nob. Same place, diff...",Not as busy as its neighbors for lunch but a d...,With two Mexican eateries on Cost Village Road...,Lagos Consumer Proxy,sha,abeg,standard
2,-FxsSuwDbIII7yo5BjHpiA,4.00,Stable,Generous,detailed,Cute Mexican place right on the corner across ...,I like Qdoba in particular for their catering ...,"The newest vessel in St. Elmo's fleet, they ha...","Well, well, well...Cunningham Restaurant Group...",If all of the fairies and pixies and good witc...,Lagos Consumer Proxy,correct,too much,NaN
3,-G7Zkl1wIWBBmD0KRy_sCw,3.67,Stable,Critical,detailed,"Just finished a State Street Kitchen, Grilled ...",Surprising good bagels from this unassuming sh...,Asian Taste Inn in the Huntingdon Valley Marke...,Came down to 40th Street to meet a good friend...,Tough crowd!\n\nCoffee depends on the quality ...,Lagos Consumer Proxy,sha,abeg,standard
4,-KduD3wwydWDq2GrODh2tw,3.50,Stable,Critical,detailed,Pretty average bar/restaurant with the menu of...,Probably the best gay bar in Indy. No smoking...,Love this store! Great selection of offbeat g...,The lunch and dinner menu is pretty standard M...,GREAT cornbread. Awesome greens. Beautifully...,Lagos Consumer Proxy,sha,abeg,standard


In [44]:
print(df.columns.tolist())
print(df.dtypes)

['record_id', 'behavioral_profile.avg_star_rating', 'behavioral_profile.rating_consistency', 'behavioral_profile.sentiment_bias', 'behavioral_profile.verbal_style', 'history[0]', 'history[1]', 'history[2]', 'history[3]', 'history[4]', 'nigerian_adaptation.persona_type', 'nigerian_adaptation.suggested_markers[0]', 'nigerian_adaptation.suggested_markers[1]', 'nigerian_adaptation.suggested_markers[2]']
record_id                                    object
behavioral_profile.avg_star_rating          float64
behavioral_profile.rating_consistency        object
behavioral_profile.sentiment_bias            object
behavioral_profile.verbal_style              object
history[0]                                   object
history[1]                                   object
history[2]                                   object
history[3]                                   object
history[4]                                   object
nigerian_adaptation.persona_type             object
nigerian_adaptation.sugge

In [45]:
def safe_parse(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        return ast.literal_eval(value)
    except Exception:
        return [str(value)]

# Parse columns
if 'history' in df.columns:
    df['history_parsed'] = df['history'].apply(safe_parse)
    df['num_reviews'] = df['history_parsed'].apply(len)
    df['avg_review_length'] = df['history_parsed'].apply(
        lambda reviews: sum(len(r) for r in reviews) / max(1, len(reviews))
    )

if 'nigerian_adaptation.suggested_markers' in df.columns:
    df['markers_parsed'] = df['nigerian_adaptation.suggested_markers'].apply(safe_parse)
    df['num_markers'] = df['markers_parsed'].apply(len)
    df['uses_sha'] = df['markers_parsed'].apply(lambda x: int('sha' in x))
    df['uses_abeg'] = df['markers_parsed'].apply(lambda x: int('abeg' in x))

In [46]:
target_column = 'behavioral_profile.avg_star_rating'

In [47]:
# Verify the target column exists
print("Available columns:")
print(df.columns.tolist())

# If the target column is missing, stop with a clear error
if target_column not in df.columns:
    raise ValueError(
        f"Target column '{target_column}' not found.\n"
        f"Available columns: {df.columns.tolist()}"
    )

# Columns to exclude from features
ignore_columns = [
    target_column,
    "history",
    "history_parsed",
    "markers_parsed"
]

# Keep only columns that are actually present
ignore_columns = [col for col in ignore_columns if col in df.columns]

# Build feature matrix and target vector
X = df.drop(columns=ignore_columns)
y = df[target_column]



Available columns:
['record_id', 'behavioral_profile.avg_star_rating', 'behavioral_profile.rating_consistency', 'behavioral_profile.sentiment_bias', 'behavioral_profile.verbal_style', 'history[0]', 'history[1]', 'history[2]', 'history[3]', 'history[4]', 'nigerian_adaptation.persona_type', 'nigerian_adaptation.suggested_markers[0]', 'nigerian_adaptation.suggested_markers[1]', 'nigerian_adaptation.suggested_markers[2]']


In [48]:
X = pd.get_dummies(X, dummy_na=True)
X = X.fillna(0)

print(X.shape)

(1017, 6127)


In [49]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% for testing
    random_state=42     # reproducible results
)

# Initialize the model
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Train the model
model.fit(X_train, y_train)


RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [51]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, min_samples_leaf=2, n_estimators=300,
                      n_jobs=-1, random_state=42)

In [52]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print('MAE:', mae)
print('RMSE:', rmse)

MAE: 0.25459057075084196
RMSE: 0.32361169665784667


In [53]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

importance.head(20)

,feature,importance
1022,behavioral_profile.sentiment_bias_Generous,0.184638
6119,nigerian_adaptation.suggested_markers[0]_correct,0.158631
6123,nigerian_adaptation.suggested_markers[1]_too much,0.152407
6126,nigerian_adaptation.suggested_markers[2]_nan,0.145980
6122,nigerian_adaptation.suggested_markers[1]_abeg,0.097327
1021,behavioral_profile.sentiment_bias_Critical,0.084344
6125,nigerian_adaptation.suggested_markers[2]_standard,0.074662
6120,nigerian_adaptation.suggested_markers[0]_sha,0.074463
1018,behavioral_profile.rating_consistency_Stable,0.011657
1019,behavioral_profile.rating_consistency_Volatile,0.011388


In [54]:
import joblib

joblib.dump(model, 'random_forest_rating.pkl')
joblib.dump(X.columns.tolist(), 'feature_columns.pkl')

['feature_columns.pkl']

In [55]:
def generate_review(pred_rating, sentiment_bias='Neutral', verbal_style='standard'):
    rating = max(1, min(5, int(round(pred_rating))))

    templates = {
        5: 'Absolutely loved this item. Highly recommended.',
        4: 'Very good overall with only minor issues.',
        3: 'It was okay, with both positives and negatives.',
        2: 'Below expectations and has several drawbacks.',
        1: 'Very disappointing and not recommended.'
    }

    review = templates[rating]

    if str(sentiment_bias).lower() == 'critical':
        review += ' Frankly, I expected more.'

    if str(verbal_style).lower() == 'detailed':
        review += ' The experience had a few notable strengths and weaknesses.'

    return review

In [56]:
df['predicted_rating'] = model.predict(X)

df['simulated_review'] = df.apply(
    lambda row: generate_review(
        row['predicted_rating'],
        row.get('behavioral_profile.sentiment_bias', 'Neutral'),
        row.get('behavioral_profile.verbal_style', 'standard')
    ),
    axis=1
)

output_cols = []
if 'user_id' in df.columns:
    output_cols.append('user_id')

output_cols += ['predicted_rating', 'simulated_review']

df[output_cols].to_csv('task_a_predictions.csv', index=False)

In [57]:
# Use predicted_rating as the preference score.
# Later, apply this model to user-item candidate pairs.

ranking = df.sort_values('predicted_rating', ascending=False)
ranking.head(10)

,record_id,behavioral_profile.avg_star_rating,behavioral_profile.rating_consistency,behavioral_profile.sentiment_bias,behavioral_profile.verbal_style,history[0],history[1],history[2],history[3],history[4],nigerian_adaptation.persona_type,nigerian_adaptation.suggested_markers[0],nigerian_adaptation.suggested_markers[1],nigerian_adaptation.suggested_markers[2],predicted_rating,simulated_review
564,WxfcSFiz7U8roEBcZ21JPA,4.40,Stable,Generous,concise,"Great pizza! Worth the wait and atmosphere, do...","Pizza was good, many many options. Awesome Gar...",Fantastic food! Definitely worth a visit. Loc...,"You just can't pass this up! Very neat, all ag...",First time in and they were great! Gel manicur...,Lagos Consumer Proxy,correct,too much,NaN,4.327638,Very good overall with only minor issues.
336,ILcfea_jMMMl-GWTaTrPUQ,4.60,Stable,Generous,concise,I'm very happy that this bike shop decided to ...,I'm not a vegetarian. I eat a paleo diet. An...,A diner in West Philly? This place does a fin...,This place has potential. It has just opened ...,This is another gem in west philly. The decor...,Lagos Consumer Proxy,correct,too much,NaN,4.327638,Very good overall with only minor issues.
71,3NYIQQCACxCdUKkWB_CnFQ,4.80,Stable,Generous,concise,Monday night have the ribs with grilled shrimp...,Very happy with Dr. Ecker. He spends time wit...,Dr Ilyas is awesome. Best dermatologist I ha...,So many horses and people at shops and food. T...,Portions are very generous I enjoyed the histo...,Lagos Consumer Proxy,correct,too much,NaN,4.327638,Very good overall with only minor issues.
945,von42BASCOHoDgA0LkRcLA,4.40,Stable,Generous,concise,They have the biggest selection of things you ...,"The Tex Mex is good, but the Margaritas are ex...",Very nice grocery store and deli. The service ...,Very consistent food and service. A personal f...,"One of our favorite spots for lunch, dinner, o...",Lagos Consumer Proxy,correct,too much,NaN,4.327638,Very good overall with only minor issues.
839,oLjseICFHuisZmxrm6bqxA,4.14,Stable,Generous,concise,Super young crowd. I guess you get what you pa...,What a fabulous restaurant! Because it was a ...,Sorry. I have to downgrade my review. While t...,This is a beautiful space for hosting private ...,Love this place. They're good at keeping appoi...,Lagos Consumer Proxy,correct,too much,NaN,4.327638,Very good overall with only minor issues.
635,bDwnxDwJU5AQUTICKeg5OQ,4.60,Stable,Generous,concise,The best hospital around! Temple has an enormo...,There isn't a bad seat anywhere in the beauty....,If you've got $6 to but awful hot sauces in a ...,Woo! Go Phillies!! \n\nThey're struggling righ...,The store gets 5 stars for convenience but 2 s...,Lagos Consumer Proxy,correct,too much,NaN,4.327638,Very good overall with only minor issues.
825,nJR5ZlxwDMx39pIbnsTFAg,4.60,Stable,Generous,concise,"I'm from STL and sadly, I never had Ted Drewes...",Amazing food! This was purchased for myself an...,"Great atmosphere, great drinks, and great food...",Great donuts and always fresh. My co-workers l...,Went here a few months ago and I was a little ...,Lagos Consumer Proxy,correct,too much,NaN,4.327638,Very good overall with only minor issues.
894,sgK8_cpe6FTk2HD0P27K0g,4.08,Stable,Generous,concise,All you can drink brunch? Yes please!\n\nSat a...,BYO mexican spot in Northern Liberties\n\nCame...,Great brunch spot in Northern Liberties. \n\nP...,The food here is pretty solid but the prices c...,Gypsy Saloon is great for Sunday brunch. For a...,Lagos Consumer Proxy,correct,too much,NaN,4.327638,Very good overall with only minor issues.
631,akUC5qP68u0_frTwHLl61w,4.33,Stable,Generous,concise,"Great beer selection. Great food, (the pizza i...",I had coffee and the pumpkin pecan hot cakes. ...,Absolute perfection. Throwback to a bygone era...,Second to Sally O'Neil's Pizza Hotline. I have...,Perfect location and beautiful view. \nRooms a...,Lagos Consumer Proxy,correct,too much,NaN,4.327638,Very good overall with only minor issues.
845,oe-CFC

In [58]:
if 'user_id' in ranking.columns:
    ranking[['user_id', 'predicted_rating']].to_csv(
        'task_b_recommendation_scores.csv', index=False
    )